In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
sys.path.append('../core')
from config import processed_data_path, results_path, raw_data_path
from model_func import save_model, save_report, save_figure, tokenize_function

import pandas as pd
import numpy as np
import torch
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding
)
from torch import nn
from datasets import Dataset, DatasetDict
from transformers import DataCollatorWithPadding

c:\Users\damitha\.conda\envs\irp_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import accelerate
import transformers
print(f"Accelerate version: {accelerate.__version__}")
print(f"Transformers version: {transformers.__version__}")

Accelerate version: 1.12.0
Transformers version: 5.1.0


In [4]:
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
MAX_LEN = 512
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-5

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

Training on device: cuda


In [6]:
structured_path = os.path.join(raw_data_path, 'ami_cohort_structured_features.csv')
nlp_features_path = os.path.join(raw_data_path, 'ami_cohort_discharge_notes.csv')

In [7]:
df_notes = pd.read_csv(nlp_features_path)
df_struct = pd.read_csv(structured_path)
df = pd.merge(df_notes, df_struct[['hadm_id', 'hospital_expire_flag']], on='hadm_id', how='inner')

In [8]:
df = df.rename(columns={'text': 'text', 'hospital_expire_flag': 'label'})
df = df[['hadm_id', 'text', 'label']].dropna()

In [9]:
df.head()

,hadm_id,text,label
0,27897940,\nName: ___ Unit No: ___\n \...,0
1,26913865,\nName: ___ Unit No: ___\n \nAdmi...,0
2,24947999,\nName: ___ Unit No: ___\n \nAdmi...,0
3,25242409,\nName: ___ Unit No: ___\n \nAdmi...,0
4,25911675,\nName: ___ Unit No: ___\n \nAdmi...,0


In [10]:
df['text'] = df['text'].astype(str)

In [11]:
print(f"Data Loaded. Shape: {df.shape}")
print(f"Class Distribution:\n{df['label'].value_counts()}")

Data Loaded. Shape: (27677, 3)
Class Distribution:
label
0    26340
1     1337
Name: count, dtype: int64


In [12]:
dataset = Dataset.from_pandas(df[['text', 'label']])

In [13]:
dataset = dataset.class_encode_column("label")

Casting to class labels: 100%|██████████| 27677/27677 [00:00<00:00, 80477.62 examples/s]


In [14]:
dataset = dataset.train_test_split(test_size=0.2, seed=42, stratify_by_column="label")

In [15]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [16]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LEN)

In [17]:
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 5536/5536 [00:17<00:00, 316.41 examples/s]


In [18]:
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

In [19]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    probs = torch.nn.functional.softmax(torch.tensor(pred.predictions), dim=-1)[:, 1].numpy()
    
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    try:
        auc = roc_auc_score(labels, probs)
    except:
        auc = 0.0
        
    return {'accuracy': acc, 'f1': f1, 'recall': recall, 'auc': auc}

In [20]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 572.14it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different ta

In [ ]:
model_save_path = os.path.join(results_path, "model/finetuned_model2")

training_args = TrainingArguments(
    output_dir=model_save_path,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE*2,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",   # <--- CHANGED THIS
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

In [22]:
train_labels = tokenized_datasets['train']['labels']

# We use standard Python sum() which works on any list/column type
num_pos = sum(train_labels)
num_neg = len(train_labels) - num_pos

# Create the tensor on the correct device
# Handle case where num_pos might be 0 to avoid crash
if num_pos > 0:
    pos_weight_value = num_neg / num_pos
else:
    pos_weight_value = 1.0

pos_weight_tensor = torch.tensor([pos_weight_value], device=device)

print(f"Class Weight Calculated: {pos_weight_value:.4f}")

Class Weight Calculated: 19.6925


In [23]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Use the global pos_weight_tensor we calculated in Step 1
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
        
        # Calculate loss
        loss = loss_fct(logits.view(-1, 2)[:, 1], labels.float())
        
        return (loss, outputs) if return_outputs else loss




In [24]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [25]:
print("Starting Fine-Tuning...")
trainer.train()

Starting Fine-Tuning...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Recall,Auc
1,0.981709,4.171041,0.951770,0.000000,0.000000,0.616015
2,6.038244,3.990250,0.951770,0.000000,0.000000,0.461814
3,4.842475,3.862829,0.951770,0.000000,0.000000,0.633255


c:\Users\damitha\.conda\envs\irp_gpu\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.06it/s]
c:\Users\damitha\.conda\envs\irp_gpu\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.02it/s]
c:\Users\damitha\.conda\envs\irp_gpu\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_div

TrainOutput(global_step=8304, training_loss=3.7674123852519115, metrics={'train_runtime': 13847.852, 'train_samples_per_second': 4.797, 'train_steps_per_second': 0.6, 'total_flos': 1.747662563017728e+16, 'train_loss': 3.7674123852519115, 'epoch': 3.0})

In [ ]:
model_path = os.path.join(results_path, "model/finetuned_clinicalbert2")
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)
print(f"Model saved to {model_path}")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.13s/it]

Model saved to E:\IIT\IRP\results\model/finetuned_clinicalbert


In [28]:
print("\nGenerating Risk Scores for full dataset...")

# Tokenize ALL data to generate the new feature column
full_dataset = Dataset.from_pandas(df[['hadm_id', 'text', 'label']])
full_tokenized = full_dataset.map(tokenize_function, batched=True)
full_tokenized = full_tokenized.remove_columns(["text", "hadm_id", "label"])


Generating Risk Scores for full dataset...


Map: 100%|██████████| 27677/27677 [01:40<00:00, 276.52 examples/s]


In [ ]:
predictions = trainer.predict(full_tokenized)
probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=-1)

In [ ]:
df['nlp_finetuned_risk_score'] = probs[:, 1].numpy()

In [ ]:
output_path =  os.path.join(processed_data_path, "nlp_finetuned_features")
df[['hadm_id', 'nlp_finetuned_risk_score']].to_csv(output_path, index=False)

NameError: name 'risk_scores' is not defined

In [ ]:
print(f"✅ Success! New features saved to: {output_path}")
print(df[['hadm_id', 'nlp_finetuned_risk_score']].head())